# 07. 바깥 밝기 추정: 규칙 vs ML

**하는 일**: 터널 입구 사진 한 장 → 바깥 밝기 L20(cd/m²)를 규칙 4개와 ML 3개 (규칙5는 11/6 결정으로 비교에서 뺌)로 추정하고 같은 자로 채점해요.
그다음 추정한 밝기 → 휘도비 → 위험도 상·중·하로 바꿔서 위험도도 채점해요.

**정답(y)**: log10(L20). L20 = 천리안 일사량 E × 277.8 × 126 × 0.15 / π (04 노트북 `data/photo_labels.csv`).
04를 아직 안 돌렸으면 임시로 ASOS 일사량으로 정답을 만들어요 (관측소가 멀어서 덜 정확, 표시해 둠).

**비교할 것 (모두 같은 사진 숫자 10개를 받음, 규칙 4·5만 예외)**

| 이름 | 하는 일 | 학습하는 것 |
|---|---|---|
| 규칙1 손 규칙 | 사람이 정한 점수표 → 점수 합 → 밝기 | 없음 (데이터 보기 전에 고정) |
| 규칙2 경계값 학습 | 점수표 모양 같음, 경계와 점수→밝기만 학습 사진에서 | 경계값 |
| 규칙3 숫자 하나 직선 | 숫자 하나로 직선 하나 (제일 잘 맞는 숫자 선택) | 기울기·절편 |
| 규칙4 물리 공식 | 맑은 하늘 공식(Haurwitz 1945) × 구름 감쇠(Kasten & Czeplak 1980). **사진 안 씀** | 없음 |
| 선형회귀 | 숫자 10개 × 가중치. 조합 못 봄 | 가중치 |
| LightGBM | 숫자 조합까지 봄 | 나무 구조 |
| LightGBM (사진+해위치·구름) | 사진 숫자 + 해 높이 + 구름 | 나무 구조 |

**시험 방법**
- 장면 단위로 나눠요 (같은 장면 사진이 학습·시험에 섞이면 안 됨). 5겹 × 10번 반복 평균
- 71786 사진이 있으면 **처음 보는 카메라 시험**: 71786으로 학습 → Mapillary로 시험 (**주 지표 = 위험도 3등급 macro-F1**, 미리 고정)
- 점수: MAE(log, 작을수록 좋음), 순위상관(클수록), 위험도F1(클수록)

순응도 A·위험도 등급 기준(10배/15배)은 **초안 · 업데이트 중 (재혁 검토 전)**. 실행 위치: `sunsight_video` 폴더.

In [ ]:
import math, json, zipfile
from pathlib import Path
import numpy as np, pandas as pd
DATA = Path('data')
SD = Path('..') / 'sunsight_data' / 'data'


## 1. 사진 숫자 뽑는 함수
노출이 바뀌어도 덜 변하는 **비율·모양** 숫자 위주. `mean_all`(평균 밝기)은 카메라 자동 노출 때문에 약할 거라 비교용으로만 넣어요.

In [ ]:
import numpy as np, cv2

def photo_features(path, tunnel_mask=None, size=640):
    """사진 한 장 → 숫자 10개. 노출이 바뀌어도 덜 변하는 '비율·모양' 숫자 위주.
    tunnel_mask: 71786처럼 터널 영역 표시가 있으면 (True=터널 구멍), 없으면 가운데 어두운 덩어리로 추정."""
    img = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)   # 한글 경로 대응
    if img is None: return None
    h, w = img.shape[:2]; s = size / max(h, w)
    img = cv2.resize(img, (int(w * s), int(h * s)))
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(float)
    H, S, V = [hsv[..., i].astype(float) for i in range(3)]
    sky, road = gray[: h // 3], gray[2 * h // 3:]
    sky_hsv = (H[: h // 3], S[: h // 3], V[: h // 3])
    blue = ((sky_hsv[0] >= 90) & (sky_hsv[0] <= 130) & (sky_hsv[1] >= 50) & (sky_hsv[2] >= 100)).mean()
    # 터널 구멍: 표시가 있으면 그대로, 없으면 가운데 영역에서 가장 어두운 15%
    if tunnel_mask is None:
        c = gray[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
        thr = np.percentile(c, 15)
        tunnel_mask = np.zeros_like(gray, bool)
        tunnel_mask[h // 4: 3 * h // 4, w // 4: 3 * w // 4] = c <= thr
    else:
        tunnel_mask = cv2.resize(tunnel_mask.astype(np.uint8), (w, h)) > 0
        if tunnel_mask.sum() > 20:   # 71786 다각형은 입구 테두리까지 덮음 → 그 안에서 가장 어두운 15% = 실제 구멍 (Mapillary와 같은 기준)
            tunnel_mask = tunnel_mask & (gray <= np.percentile(gray[tunnel_mask], 15))
    ring = cv2.dilate(tunnel_mask.astype(np.uint8), np.ones((41, 41), np.uint8)).astype(bool) & ~tunnel_mask
    portal = gray[tunnel_mask].mean() + 1 if tunnel_mask.any() else np.nan
    around = gray[ring].mean() + 1 if ring.any() else np.nan
    return {
        'sky_mean': sky.mean() / 255,                        # 하늘 밝기
        'blue_sky': blue,                                     # 파란 하늘 비율
        'white_ratio': (gray >= 250).mean(),                  # 하얗게 날아간 비율
        'shadow_contrast': (np.percentile(road, 90) - np.percentile(road, 10)) / 255,  # 그림자 대비
        'road_mean': road.mean() / 255,                       # 도로 밝기
        'sky_road_ratio': (sky.mean() + 1) / (road.mean() + 1),
        'portal_ratio': np.log10(around / portal) if portal == portal else np.nan,  # 입구/주변 밝기비(로그)
        'saturation': S.mean() / 255,                         # 색 진하기
        'spread': (np.percentile(gray, 95) - np.percentile(gray, 5)) / 255,         # 밝기 퍼짐
        'mean_all': gray.mean() / 255,                        # 평균 밝기(노출 영향 큼, 비교용)
    }
FEATURES = ['sky_mean', 'blue_sky', 'white_ratio', 'shadow_contrast', 'road_mean',
            'sky_road_ratio', 'portal_ratio', 'saturation', 'spread', 'mean_all']


## 2. 규칙 · ML · 채점 함수
정답 공식(조광 4단계, 출구 45)과 위험도 기준(10배·15배)도 여기에 있어요.

In [ ]:
# ===== 공통 상수 · 정답 공식 (05·04 노트북과 같음, 초안) =====
import numpy as np, pandas as pd, math
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from scipy.stats import spearmanr
try:
    import lightgbm as lgb
except Exception:
    lgb = None


def L20_from_E(E):                      # 일사량(MJ/m²·h) → 바깥 밝기(cd/m²)
    return E * 277.8 * 126 * 0.15 / np.pi

def ratio_from_L20(L20, cls):           # 바깥 밝기 → 휘도비 (입구는 지침 조광, 출구는 45)
    step = np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)
    return np.where(np.asarray(cls) == '출구', L20 / 45, L20 / (280 * step))

def grade(R):                           # 위험도: 10배 미만 하 / 10~15 중 / 15 초과 상 (Schreuder 1991, 초안)
    return np.select([R < 10, R <= 15], ['하', '중'], '상')

# ===== 해 높이 (NOAA 근사식) =====
def sun_altitude(ts_kst, lat, lon):
    t = pd.to_datetime(ts_kst) - pd.Timedelta(hours=9)          # UTC
    doy = t.dt.dayofyear.values; hr = (t.dt.hour + t.dt.minute / 60).values
    g = 2 * np.pi / 365 * (doy - 1 + (hr - 12) / 24)
    decl = (0.006918 - 0.399912*np.cos(g) + 0.070257*np.sin(g) - 0.006758*np.cos(2*g)
            + 0.000907*np.sin(2*g) - 0.002697*np.cos(3*g) + 0.00148*np.sin(3*g))
    eqt = 229.18 * (0.000075 + 0.001868*np.cos(g) - 0.032077*np.sin(g) - 0.014615*np.cos(2*g) - 0.040849*np.sin(2*g))
    tst = hr * 60 + eqt + 4 * np.asarray(lon)
    ha = np.radians(tst / 4 - 180); la = np.radians(np.asarray(lat))
    cosz = np.sin(la)*np.sin(decl) + np.cos(la)*np.cos(decl)*np.cos(ha)
    return np.degrees(np.arcsin(np.clip(cosz, -1, 1)))

# ===== 규칙 1: 손 규칙 (사람이 정한 점수표, 학습 없음 · 데이터 보기 전에 고정) =====
HAND = [('blue_sky', 0.20, 2), ('shadow_contrast', 0.35, 1), ('sky_mean', 0.60, 1),
        ('white_ratio', 0.01, 1), ('saturation', 0.20, 1)]       # (숫자, 경계, 점수)
def hand_score(X, cuts=None):
    s = np.zeros(len(X))
    for (f, c, p), cc in zip(HAND, cuts or [c for _, c, _ in HAND]):
        s += p * (X[f].values > cc)
    s -= 1 * (X['spread'].values < 0.5)
    return s
class Rule1Hand:
    """점수 -1~6 → 바깥 밝기 500~5,000 cd/m² (로그로 고르게). 흐림 500 · 지침 설정값 2,000~6,000을 근거로 고정."""
    name = '규칙1 손 규칙'
    def fit(self, X, y): return self
    def predict(self, X):
        s = hand_score(X); return np.log10(500) + (s + 1) / 7 * (np.log10(5000) - np.log10(500))

# ===== 규칙 2: 경계값 학습 규칙 (점수표 모양은 같고, 경계 · 점수→밝기만 학습 사진에서) =====
class Rule2Learned:
    name = '규칙2 경계값 학습'
    def fit(self, X, y):
        self.cuts = [np.median(X[f]) for f, _, _ in HAND]
        s = hand_score(X, self.cuts)
        self.table = pd.Series(y).groupby(s).mean(); self.mean = float(np.mean(y))
        self.slope = np.polyfit(s, y, 1) if len(np.unique(s)) > 1 else None
        return self
    def predict(self, X):
        s = hand_score(X, self.cuts)
        out = pd.Series(s).map(self.table).to_numpy(dtype=float, copy=True)
        miss = np.isnan(out)
        if miss.any(): out[miss] = np.polyval(self.slope, s[miss]) if self.slope is not None else self.mean
        return out

# ===== 규칙 3: 숫자 하나 직선 (숫자마다 직선 하나, 학습 사진에서 제일 잘 맞는 숫자 하나 선택) =====
class Rule3OneLine:
    name = '규칙3 숫자 하나 직선'
    def fit(self, X, y):
        best = None
        for f in FEATURES:
            x = X[f].values; ok = ~np.isnan(x)
            if ok.sum() < 5: continue
            b = np.polyfit(x[ok], y[ok], 1); e = np.mean(np.abs(np.polyval(b, x[ok]) - y[ok]))
            if best is None or e < best[0]: best = (e, f, b)
        _, self.f, self.b = best; self.mean = float(np.mean(y)); return self
    def predict(self, X):
        x = X[self.f].to_numpy(dtype=float); out = np.array(np.polyval(self.b, x)); out[np.isnan(out)] = self.mean; return out

# ===== 규칙 4: 물리 공식 (사진 안 씀: 해 높이 + 구름) =====
class Rule4Physics:
    """맑은 하늘 일사 Haurwitz(1945): 1098·cosZ·exp(-0.059/cosZ) W/m²
       × 구름 감쇠 Kasten & Czeplak(1980): 1 − 0.75·(N/8)^3.4  (ASOS 전운량 0~10을 N/8로 근사)"""
    name = '규칙4 물리 공식(사진X)'
    def fit(self, X, y): return self
    def predict(self, X):
        cosz = np.clip(np.sin(np.radians(X['sun_alt'].values)), 0.01, 1)
        G = 1098 * cosz * np.exp(-0.059 / cosz)
        c = np.nan_to_num(X['cloud'].values, nan=5) / 10
        G = G * (1 - 0.75 * c ** 3.4)
        return np.log10(np.clip(L20_from_E(G * 0.0036), 1, None))

# ===== 규칙 5: 계절 · 구름 표 (11/6 결정: 비교에서 뺌, MODELS에 안 넣음) =====
class Rule5Table:
    name = '규칙5 계절·구름 표'
    @staticmethod
    def key(X):
        season = ((X['month'].values % 12) // 3)                     # 0 겨울 1 봄 2 여름 3 가을
        cl = np.digitize(np.nan_to_num(X['cloud'].values, nan=5), [3, 8])
        return pd.Series(season * 10 + cl)
    def fit(self, X, y):
        self.table = pd.Series(y).groupby(self.key(X).values).mean(); self.mean = float(np.mean(y)); return self
    def predict(self, X):
        return self.key(X).map(self.table).fillna(self.mean).values

# ===== ML =====
class Linear:
    name = '선형회귀(조합X)'
    def fit(self, X, y):
        self.m = make_pipeline(StandardScaler(), Ridge(alpha=1.0)); self.med = X[FEATURES].median()
        self.m.fit(X[FEATURES].fillna(self.med), y); return self
    def predict(self, X): return self.m.predict(X[FEATURES].fillna(self.med))
class GBM:
    def __init__(self, extra=(), photo=True):
        self.cols = (FEATURES if photo else []) + list(extra)
        self.name = 'LightGBM' + ('' if not extra else ' (사진+해위치·구름)' if photo else ' (사진X: 해위치·구름·달)')
    def fit(self, X, y):
        # 형 PC에서 LightGBM 프로그램이 깨져(access violation) scikit-learn의 같은 방식(나무 부스팅) 모델로 바꿈. 빈칸(NaN)도 그대로 받음
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.m = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=10, random_state=0)
        self.m.fit(X[self.cols].to_numpy(dtype=np.float64), np.asarray(y, dtype=np.float64)); return self
    def predict(self, X): return self.m.predict(X[self.cols].to_numpy(dtype=np.float64))

MODELS = [Rule1Hand, Rule2Learned, Rule3OneLine, Rule4Physics, Linear, GBM,
          lambda: GBM(('sun_alt', 'cloud', 'month'), photo=False),   # 사진 없는 ML: 규칙4·5와 같은 정보 → 'ML 효과'만 따로 보기
          lambda: GBM(('sun_alt', 'cloud'))]

def scores(d, pred):
    """d: y(log L20), cls. pred: 예측 log L20 → 오차, 순위 상관, 위험도 3등급 macro-F1"""
    g_true = grade(ratio_from_L20(10 ** d.y.values, d.cls.values))
    g_pred = grade(ratio_from_L20(10 ** pred, d.cls.values))
    return {'MAE(log)': np.mean(np.abs(pred - d.y.values)),
            '순위상관': spearmanr(pred, d.y.values).correlation,
            '위험도F1': f1_score(g_true, g_pred, average='macro', labels=['하', '중', '상'], zero_division=0)}

def run_cv(d, n_rep=10, k=5):
    """장면 단위로 나눠서(같은 장면이 학습·시험에 섞이지 않게) k겹 × n_rep번 반복
    test_ok가 False인 사진(터널 없음 · 일반 도로)은 학습에만 쓰고 채점은 입구·출구 사진으로만"""
    rows = []
    ok = d.test_ok.values.astype(bool) if 'test_ok' in d else np.ones(len(d), bool)
    groups = d.scene.values; ug = np.unique(groups)
    for rep in range(n_rep):
        rng = np.random.default_rng(rep); perm = dict(zip(ug, rng.permutation(len(ug))))
        gk = np.array([perm[g] for g in groups])
        preds = {i: np.full(len(d), np.nan) for i in range(len(MODELS))}
        for tr, te in GroupKFold(k).split(d, groups=gk):
            te = te[ok[te]]
            if len(te) == 0: continue
            for i, M in enumerate(MODELS):
                m = M().fit(d.iloc[tr], d.y.values[tr]); preds[i][te] = m.predict(d.iloc[te])
        for i, M in enumerate(MODELS):
            rows.append({'모델': M().name, '반복': rep, **scores(d[ok], preds[i][ok])})
    r = pd.DataFrame(rows)
    return r.groupby('모델', sort=False)[['MAE(log)', '순위상관', '위험도F1']].agg(['mean', 'std']).round(3)

def run_camera(train, test, n_boot=1000):
    """처음 보는 카메라 시험: 71786으로 학습 → Mapillary로 시험 (주 지표)
    시험 사진이 적어서, 시험 장면을 다시 뽑아(1,000번) 위험도F1의 95% 범위도 같이 냄"""
    rows, preds = [], {}
    for M in MODELS:
        m = M().fit(train, train.y.values); preds[M().name] = m.predict(test)
        rows.append({'모델': M().name, **scores(test, preds[M().name])})
    out = pd.DataFrame(rows).set_index('모델')
    rng = np.random.default_rng(0); sc = test.scene.values; ug = np.unique(sc)
    idx = {g: np.where(sc == g)[0] for g in ug}
    boot = {k: [] for k in preds}
    for b in range(n_boot):
        ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)])
        t = test.iloc[ii]
        for k, pr in preds.items(): boot[k].append(scores(t, pr[ii])['위험도F1'])
    out['위험도F1_95%'] = [f"{np.percentile(boot[k], 2.5):.2f}~{np.percentile(boot[k], 97.5):.2f}" for k in out.index]
    return out.round(3)


## 3. 정답 읽기 (Mapillary)
- `data/photo_labels.csv`(04 결과, 천리안)가 있으면 그걸 써요
- 없으면 ASOS 일사량으로 임시 정답 (`label_source = ASOS`)
- 입구·출구는 `사진분류.csv`(05 합치기 결과)에서. 없으면 일단 전부 **입구로 가정**

In [ ]:
meta = pd.read_csv(DATA / 'tunnel_photos_meta.csv', dtype={'image_id': str}).drop_duplicates('image_id')
meta['captured_at'] = pd.to_datetime(meta.captured_at.astype(str).str[:19])

lab_path = DATA / 'photo_labels.csv'
if lab_path.exists():
    lab = pd.read_csv(lab_path, dtype={'image_id': str}).drop_duplicates('image_id')   # 한 사진이 터널 2개에 걸리면 줄이 2개 → 하나만
    lab = lab.rename(columns={'dc10Tca': 'cloud'})[['image_id', 'E', 'cloud']]
    label_source = '천리안'
else:
    st = pd.read_csv(SD / '1_원본' / 'asos_stations.csv', encoding='utf-8-sig').rename(columns={'지점': 'stn_id', '위도': 'slat', '경도': 'slon'})
    aso = pd.read_csv(SD / '1_원본' / 'asos_hourly_all.csv', usecols=['stn_id', 'tm', 'icsr', 'dc10Tca'], parse_dates=['tm'])
    meas = aso.groupby('stn_id').icsr.max(); st = st[st.stn_id.isin(meas[meas > 0].index)].reset_index(drop=True)
    def near(lat, lon):
        d = np.hypot(st.slat - lat, (st.slon - lon) * math.cos(math.radians(lat))); return st.stn_id[d.idxmin()]
    m = meta.copy(); m['stn_id'] = [near(a, b) for a, b in zip(m.lat, m.lon)]; m['tm'] = m.captured_at.dt.ceil('h')
    lab = m.merge(aso, on=['stn_id', 'tm'], how='left').rename(columns={'icsr': 'E', 'dc10Tca': 'cloud'})[['image_id', 'E', 'cloud']]
    label_source = 'ASOS (임시)'
print('정답 출처:', label_source)

d = meta.merge(lab, on='image_id', how='left')
# 분류 결과: 05 합친 파일이 있으면 그것, 없으면 sunsight_data/사진분류_*.csv 들 (여러 번 분류했으면 답이 같은 것만)
cls_files = [DATA / '사진분류.csv'] if (DATA / '사진분류.csv').exists() else sorted(SD.parent.glob('사진분류_*.csv'))
if cls_files:
    cl = pd.concat([pd.read_csv(f, dtype=str, encoding='utf-8-sig')[['image_id', 'class']] for f in cls_files])
    agree = cl.groupby('image_id')['class'].agg(lambda s: s.iloc[0] if s.nunique() == 1 else '불일치')
    d = d.merge(agree.rename('class').reset_index(), on='image_id', how='left')
    d['class'] = d['class'].fillna('미분류')
    print('분류 파일', [f.name for f in cls_files], '→', d['class'].value_counts().to_dict())
    d = d[~d['class'].isin(['터널 안', '불일치'])]          # 터널 안 = 하늘이 안 보임
    d['test_ok'] = d['class'].isin(['입구', '출구'])       # 채점은 입구·출구만. 터널 없음 · 미분류는 학습에만
    d['cls'] = np.where(d['class'] == '출구', '출구', '입구')
else:
    d['cls'] = '입구'; d['test_ok'] = True; print('분류 파일 없음 → 전부 입구로 가정 (분류 후 다시 실행)')

sc = pd.read_csv(DATA / '사진_중복묶음.csv', dtype={'image_id': str})[['image_id', '장면번호']]
d = d.merge(sc.rename(columns={'장면번호': 'scene'}), on='image_id', how='left')
d['scene'] = d.scene.fillna(-1).astype(int)
d = d[~d.tunnel.astype(str).str.contains('지하차도')]
d = d[d.E > 0.05].copy()                                   # 밤 · 해 질 녘 제외
d['y'] = np.log10(L20_from_E(d.E))
d['sun_alt'] = sun_altitude(d.captured_at, d.lat, d.lon)
d['month'] = d.captured_at.dt.month
d['source'] = 'mapillary'
print(len(d), '장 ·', d.scene.nunique(), '장면')
print('정답 위험도 분포:', pd.Series(grade(ratio_from_L20(10 ** d.y.values, d.cls.values))).value_counts().to_dict())


## 4. 사진 숫자 뽑기 (Mapillary)

In [ ]:
rows = []
for r in d.itertuples():
    f = photo_features(DATA / 'tunnel_photos' / f'{r.image_id}.jpg')
    if f: f['image_id'] = r.image_id; rows.append(f)
d = d.merge(pd.DataFrame(rows), on='image_id')
d.loc[~d.test_ok, 'portal_ratio'] = np.nan                 # 입구가 안 보이는 사진은 입구 숫자를 빈칸으로

# 일반 도로 사진 (08 노트북 결과, 있을 때만): 학습에만 추가
road_path = DATA / 'road_photos_train.csv'
if road_path.exists():
    rd = pd.read_csv(road_path, dtype={'image_id': str}, parse_dates=['captured_at'])
    rd['y'] = np.log10(L20_from_E(rd.E)); rd['sun_alt'] = sun_altitude(rd.captured_at, rd.lat, rd.lon)
    rd['cls'] = '입구'; rd['test_ok'] = False; rd['source'] = 'road'
    # 같은 여정(시험 사진과 같은 날 2시간 안) 도로 사진: 정답(시간 단위 일사량)이 거의 같아 새는 길 → 장면 번호를 그 시험 사진과 묶고, ②b 학습에서는 뺌
    tt = d[d.test_ok][['captured_at', 'scene']].copy(); rd['같은여정'] = False
    for i, r in rd.iterrows():
        m = tt[(tt.captured_at.dt.date == r.captured_at.date()) & ((tt.captured_at - r.captured_at).abs() <= pd.Timedelta(hours=2))]
        if len(m): rd.at[i, 'scene'] = m.scene.iloc[0]; rd.at[i, '같은여정'] = True
    print('시험 사진과 같은 여정인 도로 사진', int(rd['같은여정'].sum()), '/', len(rd), '장 → 장면 묶음 · ②b 학습 제외')
    d = pd.concat([d, rd[[c for c in d.columns if c in rd.columns] + ['같은여정']]], ignore_index=True)
print('학습 가능', len(d), '장 | 채점용(입구·출구)', int(d.test_ok.sum()), '장')
d[FEATURES].describe().T.round(3)


## 5. 71786 사진 (있을 때만)
- 06 노트북(터널 사진 고르기)으로 받은 사진을 `data/aihub_71786/photos/`에, 고른 목록을 `data/aihub_71786/photos.csv`(file_name, class, E, cloud, lat, lon, day, time)에 두면 같이 써요
- 71786은 라벨에 터널 영역(다각형)이 있어서 입구/주변 밝기비를 그 영역으로 계산해요
- 없으면 이 칸은 건너뛰고 Mapillary만으로 시험해요

In [ ]:
A71 = DATA / 'aihub_71786'
LABEL_ZIPS = sorted((SD / '71786_라벨').glob('*.zip'))
d71 = None
if (A71 / 'photos.csv').exists():
    p = pd.read_csv(A71 / 'photos.csv')
    want = set(p.file_name)
    polys = {}
    for zp in LABEL_ZIPS:
        z = zipfile.ZipFile(zp)
        for n in z.namelist():
            fn = Path(n).stem + '.jpg'
            if fn not in want: continue
            j = json.loads(z.read(n).decode('utf-8-sig'))
            tid = [c['id'] for c in j['categories'] if c['name'] == '터널']
            seg = [a['segmentation'] for a in j['annotations'] if a.get('category_id') in tid]
            polys[fn] = (seg, j['images'][0]['width'], j['images'][0]['height'])
    rows = []
    for r in p.itertuples():
        seg, W, H = polys.get(r.file_name, ([], 0, 0))
        mask = None
        if seg:
            mask = np.zeros((H, W), np.uint8)
            for s in seg:
                pts = np.array(s, float).reshape(-1, 2).astype(np.int32); cv2.fillPoly(mask, [pts], 1)
        f = photo_features(A71 / 'photos' / r.file_name, tunnel_mask=mask)
        if f: f['file_name'] = r.file_name; rows.append(f)
    d71 = p.merge(pd.DataFrame(rows), on='file_name')
    d71['captured_at'] = pd.to_datetime('20' + d71.day.astype(str) + ' ' + d71.time.astype(str))
    d71 = d71[d71.E > 0.05].copy()
    d71['y'] = np.log10(L20_from_E(d71.E)); d71['cls'] = d71['class']
    d71['sun_alt'] = sun_altitude(d71.captured_at, d71.lat, d71.lon); d71['month'] = d71.captured_at.dt.month
    if 'scene' not in d71: d71['scene'] = d71.groupby(['day', 'lat', 'lon']).ngroup() + 100000   # 06에서 만든 장면(2분·1 km) 우선
    d71['source'] = '71786'
    d71['test_ok'] = d71['class'].isin(['입구', '출구'])
    d71.loc[~d71.test_ok, 'portal_ratio'] = np.nan                      # 일반 도로 사진: 입구 숫자 빈칸
    d71 = d71[d71.test_ok | (d71.mean_all >= 0.12)].copy()              # 일반 도로 중 너무 어두운 사진(터널 안일 수 있음) 제외
    print('71786:', len(d71), '장')
    d71.drop(columns=['captured_at'], errors='ignore').to_csv(A71 / '학습표_71786.csv', index=False, encoding='utf-8-sig')   # 09(블랙박스 시험)가 씀
else:
    print('71786 사진 없음 → Mapillary만으로 시험')


## 6. 시험 ① 장면 단위 5겹 × 10번 (같은 카메라 안에서)

In [ ]:
cv_map = run_cv(d, n_rep=10)
print('Mapillary,', label_source, ',', len(d), '장'); cv_map


## 7. 시험 ② 처음 보는 카메라 (주 지표) — 71786으로 학습 → Mapillary로 시험

In [ ]:
if d71 is not None:
    cam = run_camera(d71, d[d.test_ok])          # 시험은 Mapillary 입구·출구 사진만
    display(cam)
    print('71786 안에서 장면 단위 시험 (같은 카메라, 참고)'); cv71 = run_cv(d71, n_rep=5); display(cv71)
else:
    cam = None; print('71786 사진이 생기면 다시 실행')


In [ ]:
# ===== 이김 기준 ② 짝지은 차이 (11/5 추가) =====
# 71786으로 학습 → Mapillary 입구·출구로 시험. 같은 시험 장면을 1,000번 다시 뽑아 "ML F1 − 상대 F1"의 95% 범위
# 이김 (11/5 형 결정, 모델②와 같은 기준) = 장면 다시 뽑기 F1 차이의 95% 범위가 0을 포함하지 않음(하한 > 0). 이전 엄격 기준(F1+.05 · 순위상관·MAE 같은 방향)은 참고로 함께 표시
test = d[d.test_ok]; P = {}
for M in MODELS:
    m = M(); P[m.name] = m.fit(d71, d71.y.values).predict(test)
rules = [k for k in P if k.startswith('규칙')]; mls = [k for k in P if not k.startswith('규칙')]
rng = np.random.default_rng(0); sc = test.scene.values; ug = np.unique(sc)
idx = {g: np.where(sc == g)[0] for g in ug}
B = []
for b in range(1000):
    ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)]); t = test.iloc[ii]
    B.append({k: scores(t, P[k][ii])['위험도F1'] for k in P})
B = pd.DataFrame(B); full = {k: scores(test, P[k])['위험도F1'] for k in P}
rows = []
for a in mls:
    for r in rules + [x for x in mls if x != a]:
        diff = B[a] - B[r]
        rows.append({'ML': a, '상대': r, 'F1 차이': round(full[a] - full[r], 3),
                     '95% 하한': round(diff.quantile(.025), 3), '95% 상한': round(diff.quantile(.975), 3)})
pair = pd.DataFrame(rows); pair['이김 (팀 공통 기준: 95% 하한>0)'] = pair['95% 하한'] > 0; pair['(이전 엄격 기준) F1+.05 & 하한>0'] = (pair['F1 차이'] >= .05) & (pair['95% 하한'] > 0)
pair.to_csv(DATA / '밝기추정_짝차이.csv', index=False, encoding='utf-8-sig')
print('시험 사진', len(test), '장 ·', len(ug), '장면 · 저장: data/밝기추정_짝차이.csv'); pair


In [ ]:
# ===== 보조 지표: 경고 관점 재현율 (11/5 형 요청, 주 지표 F1은 그대로) =====
# 상 재현율 = 진짜 '상'인 사진 중 '상'으로 경고한 비율. 경고(중+상) 재현율 = 진짜 중·상 중 중·상으로 낸 비율
# 늘 '상'만 내면 재현율 1이 되므로 정밀도(경고한 것 중 진짜 비율)와 같이 봄
def warn_scores(t, pred):
    gt = grade(ratio_from_L20(10 ** t.y.values, t.cls.values)); gp = grade(ratio_from_L20(10 ** pred, t.cls.values))
    w_t, w_p = gt != '하', gp != '하'
    sr = lambda a, b: round(float((a & b).sum() / max(b.sum(), 1)), 3)
    return {'상 재현율': sr(gp == '상', gt == '상'), '상 정밀도': sr(gt == '상', gp == '상'),
            '경고 재현율': sr(w_p, w_t), '경고 정밀도': sr(w_t, w_p), '진짜 상': int((gt == '상').sum()), '진짜 중·상': int(w_t.sum())}
rec = pd.DataFrame({k: warn_scores(test, P[k]) for k in P}).T
print('정답 등급 분포 (시험):', pd.Series(grade(ratio_from_L20(10 ** test.y.values, test.cls.values))).value_counts().to_dict())
display(rec)

# ===== 하이퍼파라미터 튜닝: 71786 안에서만 고름 (Mapillary 시험 사진은 고를 때 안 봄 = 누수 없음) =====
# 고르는 기준 = 71786 장면 단위 5겹 MAE (71786은 8~9월 맑은 날이 많아 F1이 거의 안 변하므로 밝기 오차로 고름)
from sklearn.ensemble import HistGradientBoostingRegressor
from itertools import product
GRID = list(product([0.03, 0.1], [7, 15, 31], [100, 300], [10, 40], [0.0, 1.0]))   # 학습률 · 잎 수 · 나무 수 · 잎 최소 사진 · L2 = 48개
SETS = {'사진만': FEATURES, '사진X': ['sun_alt', 'cloud', 'month'], '사진+': FEATURES + ['sun_alt', 'cloud']}
def hgb(prm): return HistGradientBoostingRegressor(learning_rate=prm[0], max_leaf_nodes=prm[1], max_iter=prm[2],
                                                   min_samples_leaf=prm[3], l2_regularization=prm[4], random_state=0)
ok71 = d71.test_ok.values.astype(bool); folds = list(GroupKFold(5).split(d71, groups=d71.scene.values))
tune = []; PT = {}
for nm, cols in SETS.items():
    X71 = d71[cols].to_numpy(dtype=np.float64); best = None
    for prm in GRID:
        pr = np.full(len(d71), np.nan)
        for tr, te in folds:
            pr[te] = hgb(prm).fit(X71[tr], d71.y.values[tr]).predict(X71[te])
        mae = np.mean(np.abs(pr[ok71] - d71.y.values[ok71]))
        if best is None or mae < best[0]: best = (mae, prm)
    m = hgb(best[1]).fit(X71, d71.y.values); pt = m.predict(test[cols].to_numpy(dtype=np.float64)); PT['부스팅 ' + nm + ' (튜닝)'] = pt
    tune.append({'모델': '부스팅 ' + nm + ' (튜닝)', '고른 값(학습률·잎·나무·최소·L2)': best[1], '71786 CV MAE': round(best[0], 3),
                 **{k: round(v, 3) for k, v in scores(test, pt).items()}, **warn_scores(test, pt)})
    print(nm, '끝')
tune = pd.DataFrame(tune).set_index('모델')
pd.concat([rec, tune]).to_csv(DATA / '밝기추정_재현율_튜닝.csv', encoding='utf-8-sig')
print('튜닝 전 F1:', {k: round(full[k], 3) for k in mls}); print('저장: data/밝기추정_재현율_튜닝.csv'); tune


In [ ]:
# ===== 같은 계절만 시험 (11/5): 점수가 떨어진 이유가 "카메라"인지 "계절"인지 가르기 =====
# 71786 학습은 8~9월뿐 → Mapillary 시험도 6~9월(해 강한 철)만 남겨 채점. 여기서도 낮으면 카메라 탓, 오르면 계절 탓
sm = test.month.between(6, 9).values; ts = test[sm]
print('6~9월 시험 사진', len(ts), '장 ·', ts.scene.nunique(), '장면 | 정답 등급', pd.Series(grade(ratio_from_L20(10 ** ts.y.values, ts.cls.values))).value_counts().to_dict())
season = pd.DataFrame([{'모델': k, **{a: round(b, 3) for a, b in scores(ts, P[k][sm]).items()}, **warn_scores(ts, P[k][sm])} for k in P]).set_index('모델')
season.to_csv(DATA / '밝기추정_같은계절.csv', encoding='utf-8-sig'); display(season)

# ===== 튜닝한 모델도 짝지은 차이로 (같은 1,000번 다시 뽑기) =====
rng = np.random.default_rng(0); rows = []
BT = []
for b in range(1000):
    ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)]); t = test.iloc[ii]
    BT.append({k: scores(t, v[ii])['위험도F1'] for k, v in {**PT, **{r: P[r] for r in rules}}.items()})
BT = pd.DataFrame(BT)
for a in PT:
    for r in rules:
        dd = BT[a] - BT[r]; fd = scores(test, PT[a])['위험도F1'] - full[r]
        rows.append({'ML': a, '상대': r, 'F1 차이': round(fd, 3), '95% 하한': round(dd.quantile(.025), 3), '95% 상한': round(dd.quantile(.975), 3)})
tpair = pd.DataFrame(rows); tpair['이김 (팀 공통 기준: 95% 하한>0)'] = tpair['95% 하한'] > 0; tpair['(이전 엄격 기준) F1+.05 & 하한>0'] = (tpair['F1 차이'] >= .05) & (tpair['95% 하한'] > 0)
tpair.to_csv(DATA / '밝기추정_튜닝_짝차이.csv', index=False, encoding='utf-8-sig'); display(tpair)

# ===== 주 모델 고르기: 튜닝한 ML끼리 짝지은 차이 (사진+ vs 사진X 등) =====
rows = []; ks = list(PT)
for i, a in enumerate(ks):
    for bnm in ks[i+1:]:
        fa = scores(test, PT[a])['위험도F1']; fb = scores(test, PT[bnm])['위험도F1']
        hi, lo = (a, bnm) if fa >= fb else (bnm, a)
        dd = BT[hi] - BT[lo]
        rows.append({'높은 쪽': hi, '낮은 쪽': lo, 'F1 차이': round(abs(fa - fb), 3), '95% 하한': round(dd.quantile(.025), 3), '95% 상한': round(dd.quantile(.975), 3)})
mlpair = pd.DataFrame(rows); mlpair['이김 (팀 공통 기준: 95% 하한>0)'] = mlpair['95% 하한'] > 0; mlpair['(이전 엄격 기준) F1+.05 & 하한>0'] = (mlpair['F1 차이'] >= .05) & (mlpair['95% 하한'] > 0)
mlpair.to_csv(DATA / '밝기추정_튜닝_ML짝차이.csv', index=False, encoding='utf-8-sig'); display(mlpair)

# ===== 세 번째 조건 확인: 순위상관·MAE 차이도 장면 다시 뽑기 1,000번 (사진+ vs 사진X) =====
a, bnm = '부스팅 사진+ (튜닝)', '부스팅 사진X (튜닝)'
rng = np.random.default_rng(0); yv = test.y.values; R = []
for b in range(1000):
    ii = np.concatenate([idx[g] for g in rng.choice(ug, len(ug), replace=True)])
    R.append({'순위상관 차이': spearmanr(PT[a][ii], yv[ii]).correlation - spearmanr(PT[bnm][ii], yv[ii]).correlation,
              'MAE 차이': np.mean(np.abs(PT[a][ii] - yv[ii])) - np.mean(np.abs(PT[bnm][ii] - yv[ii]))})
R = pd.DataFrame(R)
rk = pd.DataFrame({'차이(사진+ − 사진X)': [spearmanr(PT[a], yv).correlation - spearmanr(PT[bnm], yv).correlation, np.mean(np.abs(PT[a] - yv)) - np.mean(np.abs(PT[bnm] - yv))],
                   '95% 하한': R.quantile(.025).values, '95% 상한': R.quantile(.975).values}, index=['순위상관', 'MAE(log)']).round(3)
rk.to_csv(DATA / '밝기추정_튜닝_순위상관차이.csv', encoding='utf-8-sig'); display(rk)


In [ ]:
# ===== ②b 겨울 보충 시험 (11/5): 71786 + 08 터널 없는 도로 사진(Mapillary, 여러 달)으로 학습 → 같은 Mapillary 입구·출구로 시험 =====
# 08 사진은 터널에서 떨어진 도로라 시험 사진(터널 입구·출구)과 겹치지 않음. 학습에만 씀
road = d[(d.source == 'road') & (d.E > 0.05)] if 'source' in d else d.iloc[:0]
if '같은여정' in road: road = road[~road['같은여정'].fillna(False).astype(bool)]   # 시험과 같은 여정 사진 제외 (11/5)
if len(road):
    tr_b = pd.concat([d71, road[[c for c in d71.columns if c in road.columns]]], ignore_index=True)
    print('②b 학습', len(tr_b), '장 (71786', len(d71), '+ 도로', len(road), ') | 학습 달 분포', tr_b.month.value_counts().sort_index().to_dict())
    cam_b = run_camera(tr_b, test)
    PB = {M().name: M().fit(tr_b, tr_b.y.values).predict(test) for M in MODELS}
    cam_b = cam_b.join(pd.DataFrame({k: warn_scores(test, v) for k, v in PB.items()}).T[['상 재현율', '상 정밀도', '경고 재현율', '경고 정밀도']])
    cam_b['시험'] = 'Mapillary 전부'
    cam_b.to_csv(DATA / '밝기추정_2b.csv', encoding='utf-8-sig'); display(cam_b)
    # 같은 학습 모델을 Mapillary 6~9월 사진만으로도 채점 (해 강한 철)
    cam_b2 = pd.DataFrame([{'모델': k, **{x: round(y, 3) for x, y in scores(ts, v[sm]).items()}, **warn_scores(ts, v[sm])} for k, v in PB.items()]).set_index('모델')
    cam_b2['시험'] = 'Mapillary 6~9월'; cam_b2.to_csv(DATA / '밝기추정_2b_같은계절.csv', encoding='utf-8-sig')
    print('②b · 6~9월만 채점 (', len(ts), '장)'); display(cam_b2)
else:
    print('08 결과(road_photos_train.csv)가 없음 → 08 실행 후 다시')


In [ ]:
# ===== 동공 모델로 넘길 사진별 밝기 (11/5): 동공 스레드와 맞춘 형식 =====
# 주 모델 후보(튜닝한 사진+ 부스팅)와 비교용(물리 공식)의 예측을 사진마다 저장. 시험(Mapillary 입구·출구) 사진만
def step_of(L20): return np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)
def inside_L(L20, cls): return np.where(np.asarray(cls) == '출구', 45.0, 280 * step_of(L20))
rows = {'image_id': test.image_id.values, 'source': test.source.values, 'cls': test.cls.values,
        'captured_at': test.captured_at.values, 'month': test.month.values, 'sun_alt': test.sun_alt.values.round(1),
        'L20_정답': (10 ** test.y.values).round(1)}
for tag, pr in [('사진+튜닝', PT.get('부스팅 사진+ (튜닝)')), ('물리공식', P['규칙4 물리 공식(사진X)'])]:
    if pr is None: continue
    L = 10 ** pr; Lin = inside_L(L, test.cls.values)
    # 출구는 안→밖: 동공 모델에 넣을 "들어가기 전·후 밝기"를 방향대로 적음
    before = np.where(test.cls.values == '출구', Lin, L); after = np.where(test.cls.values == '출구', L, Lin)
    rows[f'L20_{tag}'] = L.round(1); rows[f'안밝기_{tag}'] = np.round(Lin, 1)
    rows[f'전밝기_{tag}'] = np.round(before, 1); rows[f'후밝기_{tag}'] = np.round(after, 1)
    rows[f'휘도비_{tag}'] = np.round(L / Lin, 2); rows[f'위험도_{tag}'] = grade(L / Lin)
Lt = 10 ** test.y.values; Lint = inside_L(Lt, test.cls.values)
rows['안밝기_정답'] = np.round(Lint, 1); rows['휘도비_정답'] = np.round(Lt / Lint, 2); rows['위험도_정답'] = grade(Lt / Lint)
exp = pd.DataFrame(rows); exp.to_csv(DATA / '밝기추정_사진별.csv', index=False, encoding='utf-8-sig')
print('저장 data/밝기추정_사진별.csv', len(exp), '장 (cd/m², 출구는 안→밖 순서로 전·후 밝기)'); exp.head()


## 8. 저장 · 읽는 법
- `data/밝기추정_결과.csv`에 저장
- **사진만 쓰는 줄끼리** (규칙1~3 vs 선형 vs LightGBM): ML이 숫자 조합으로 얼마나 더 맞히나
- **규칙4·5 vs LightGBM(사진+해위치·구름)**: 사진이 날씨·시각 정보에 무엇을 더 보태나
- 주장은 처음 보는 카메라 결과로 해요 (같은 카메라 안에서는 규칙2가 ML과 비슷할 수 있음, 모델② 경험)

In [ ]:
out = cv_map.copy(); out.columns = ['_'.join(c) for c in out.columns]; out['시험'] = f'Mapillary 장면CV ({label_source})'
if cam is not None:
    c2 = cam.copy(); c2['시험'] = '71786→Mapillary'; out = pd.concat([out, c2])
    c3 = cv71.copy(); c3.columns = ['_'.join(c) for c in c3.columns]; c3['시험'] = '71786 장면CV'; out = pd.concat([out, c3])
out.to_csv(DATA / '밝기추정_결과.csv', encoding='utf-8-sig')
print('저장: data/밝기추정_결과.csv')


In [ ]:
# ===== 입구(암순응)만 채점 (11/5 형 결정 검토: 출구 사진 적고 71786 학습엔 0장 → 주 판정 후보) =====
ent = (test.cls == '입구').values; te = test[ent]
ALL = {**P, **{k: v for k, v in PT.items()}}
if 'PB' in globals(): ALL.update({'②b ' + k: v for k, v in PB.items()})
ent_tab = pd.DataFrame([{'모델': k, **{a: round(b, 3) for a, b in scores(te, v[ent]).items()}, **warn_scores(te, v[ent])} for k, v in ALL.items()]).set_index('모델')
ent_tab['시험'] = f'Mapillary 입구만 ({len(te)}장)'
ent_tab.to_csv(DATA / '밝기추정_입구만.csv', encoding='utf-8-sig'); display(ent_tab)
# 짝지은 차이 (입구 장면 다시 뽑기 1,000번)
rng = np.random.default_rng(0); sce = te.scene.values; uge = np.unique(sce); ide = {g: np.where(sce == g)[0] for g in uge}
BE = []
for b in range(1000):
    ii = np.concatenate([ide[g] for g in rng.choice(uge, len(uge), replace=True)]); t = te.iloc[ii]
    BE.append({k: scores(t, v[ent][ii])['위험도F1'] for k, v in ALL.items()})
BE = pd.DataFrame(BE); fe = {k: scores(te, v[ent])['위험도F1'] for k, v in ALL.items()}
rows = []
for a in [k for k in ALL if not k.split(' ')[-1].startswith('규칙') and not k.startswith('규칙')]:
    for r in rules:
        dd = BE[a] - BE[r]
        rows.append({'ML': a, '상대': r, 'F1 차이': round(fe[a] - fe[r], 3), '95% 하한': round(dd.quantile(.025), 3), '95% 상한': round(dd.quantile(.975), 3)})
epair = pd.DataFrame(rows); epair['이김 (팀 공통 기준: 95% 하한>0)'] = epair['95% 하한'] > 0; epair['(이전 엄격 기준) F1+.05 & 하한>0'] = (epair['F1 차이'] >= .05) & (epair['95% 하한'] > 0)
epair.to_csv(DATA / '밝기추정_입구만_짝차이.csv', index=False, encoding='utf-8-sig'); display(epair)
